In [ ]:
import copy
import os
import sys
import time
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from argparse import Namespace
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.nn import functional as F

try:
    repo_root = os.path.dirname(os.path.abspath(__file__))
except NameError:
    repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.model import generate_model, _modify_first_conv_layer
from src.mean import get_mean, get_std
from src.setup import resolve_dataset_paths
from src.dataset import get_test_set
from src.transforms.spatial_transforms import Compose, Scale, CenterCrop, ToTensor, Normalize
from src.transforms.temporal_transforms import TemporalCenterCrop
from src.transforms.target_transforms import ClassLabel
from src.utils import Queue

# Configuration
opt = Namespace(
    # General Options
    no_cuda=not torch.cuda.is_available(),
    norm_value=1,
    batch_size=1,
    manual_seed=1,
    n_threads=0, # Set to 0 for Jupyter compatibility 
    
    # Dataset Paths
    dataset='ipn',
    ipn_root_path=repo_root,
    ipn_video_path='src/datasets/HandGestures/IPN_dataset',
    ipn_annotation_path='annotation_ipnGesture/ipnall_but_None.json',
    jester_root_path='', jester_video_path='', jester_annotation_path='',
    
    # Detector Configuration (CNN Path)
    det_backend='cnn', # Change to 'mediapipe' for lightweight inference
    resume_path_det=os.path.join(repo_root, 'results_ipn/ipnDet_sc8b64_resnetl-10_best.pth'),
    sample_duration_det=8,
    model_det='resnetl',
    model_depth_det=10,
    resnet_shortcut_det='A',
    modality_det='RGB',
    n_classes_det=2,
    n_finetune_classes_det=2,
    no_first_lay_det=False,
    
    # Classifier Configuration
    resume_path_clf=os.path.join(repo_root, 'results_ipn/ipnClf_jes32r_b32_resnext-101_best.pth'),
    sample_duration_clf=32,
    model_clf='resnext',
    model_depth_clf=101,
    resnet_shortcut_clf='B',
    modality_clf='RGB',
    n_classes_clf=13,
    n_finetune_classes_clf=13,
    no_first_lay_clf=False,
    
    # Preprocessing
    mean_dataset='ipn',
    no_mean_norm=False,
    std_norm=False,
    sample_size=112,
    
    # Smoothing / Inference logic
    det_strategy='ma',
    det_queue_size=4,
    det_counter=2,
    clf_strategy='ma',
    clf_queue_size=16,
    clf_threshold_pre=0.15,
    clf_threshold_final=0.15,
    
    # Placeholders for underlying engine functions
    pretrain_path='', pretrain_path_det='', pretrain_path_clf='', resume_path='',
    scales=[1.0], n_scales=5, scale_step=0.84089641525, initial_scale=1.0,
    store_name='RGB', test_subset='test', true_valid=False
)

In [ ]:
# Initialize paths and spatial transformations
resolve_dataset_paths(opt)
opt.mean = get_mean(opt.norm_value)
opt.std = get_std(opt.norm_value)

spatial_transform = Compose([
    Scale(opt.sample_size),
    CenterCrop(opt.sample_size),
    ToTensor(opt.norm_value),
    Normalize(opt.mean, opt.std),
])

# Load Models
def _get_smoothed(queue, strategy):
    return queue.ewma if strategy == 'ewma' else queue.ma

def load_cnn_detector(opt):
    o = copy.copy(opt)
    o.resume_path = opt.resume_path_det
    o.sample_duration = opt.sample_duration_det
    o.model = opt.model_det
    o.model_depth = opt.model_depth_det
    o.modality = opt.modality_det
    o.resnet_shortcut = opt.resnet_shortcut_det
    o.n_classes = opt.n_classes_det
    o.n_finetune_classes = opt.n_finetune_classes_det
    o.no_first_lay = opt.no_first_lay_det
    o.arch = f'{o.model}-{o.model_depth}'
    o.pretrain_path = getattr(opt, 'pretrain_path', '')

    detector, _ = generate_model(o)

    if os.path.exists(o.resume_path):
        checkpoint = torch.load(o.resume_path, map_location='cpu', weights_only=False)
        if 'module.conv1.weight' in checkpoint['state_dict']:
            ckpt_w = checkpoint['state_dict']['module.conv1.weight']
            m = detector.module if hasattr(detector, 'module') else detector
            if ckpt_w.shape[1] != m.conv1.weight.shape[1] or ckpt_w.shape[2] != m.conv1.weight.shape[2]:
                detector = _modify_first_conv_layer(detector, ckpt_w.shape[2], ckpt_w.shape[1])
        detector.load_state_dict(checkpoint['state_dict'], strict=False)

    detector.eval()
    if not opt.no_cuda: detector = detector.cuda()
    return detector

# Initialize paths and spatial transformations
resolve_dataset_paths(opt)
opt.mean = get_mean(opt.norm_value)
opt.std = get_std(opt.norm_value)

spatial_transform = Compose([
    Scale(opt.sample_size),
    CenterCrop(opt.sample_size),
    ToTensor(opt.norm_value),
    Normalize(opt.mean, opt.std),
])

# Load Models
def _get_smoothed(queue, strategy):
    return queue.ewma if strategy == 'ewma' else queue.ma

def load_cnn_detector(opt):
    o = copy.copy(opt)
    o.resume_path = opt.resume_path_det
    o.sample_duration = opt.sample_duration_det
    o.model = opt.model_det
    o.model_depth = opt.model_depth_det
    o.modality = opt.modality_det
    o.resnet_shortcut = opt.resnet_shortcut_det
    o.n_classes = opt.n_classes_det
    o.n_finetune_classes = opt.n_finetune_classes_det
    o.no_first_lay = opt.no_first_lay_det
    o.arch = f'{o.model}-{o.model_depth}'

    detector, _ = generate_model(o)

    if os.path.exists(o.resume_path):
        checkpoint = torch.load(o.resume_path, map_location='cpu', weights_only=False)
        if 'module.conv1.weight' in checkpoint['state_dict']:
            ckpt_w = checkpoint['state_dict']['module.conv1.weight']
            m = detector.module if hasattr(detector, 'module') else detector
            if ckpt_w.shape[1] != m.conv1.weight.shape[1] or ckpt_w.shape[2] != m.conv1.weight.shape[2]:
                detector = _modify_first_conv_layer(detector, ckpt_w.shape[2], ckpt_w.shape[1])
        detector.load_state_dict(checkpoint['state_dict'], strict=False)

    detector.eval()
    if not opt.no_cuda: detector = detector.cuda()
    return detector



In [4]:
def load_classifier(opt):
    o = copy.copy(opt)
    o.resume_path = opt.resume_path_clf
    o.sample_duration = opt.sample_duration_clf
    o.model = opt.model_clf
    o.model_depth = opt.model_depth_clf
    o.modality = opt.modality_clf
    o.resnet_shortcut = opt.resnet_shortcut_clf
    o.n_classes = opt.n_classes_clf
    o.n_finetune_classes = opt.n_finetune_classes_clf
    o.no_first_lay = opt.no_first_lay_clf
    o.arch = f'{o.model}-{o.model_depth}'

    classifier, _ = generate_model(o)

    if os.path.exists(o.resume_path):
        checkpoint = torch.load(o.resume_path, map_location='cpu', weights_only=False)
        if 'module.conv1.weight' in checkpoint['state_dict']:
            ckpt_w = checkpoint['state_dict']['module.conv1.weight']
            m = classifier.module if hasattr(classifier, 'module') else classifier
            if ckpt_w.shape[1] != m.conv1.weight.shape[1] or ckpt_w.shape[2] != m.conv1.weight.shape[2]:
                classifier = _modify_first_conv_layer(classifier, ckpt_w.shape[2], ckpt_w.shape[1])
        classifier.load_state_dict(checkpoint['state_dict'], strict=False)

    classifier.eval()
    if not opt.no_cuda: classifier = classifier.cuda()
    return classifier


In [8]:
print("Loading Detector...")
if opt.det_backend == 'mediapipe':
    from src.mediapipe_detector import MediaPipeDetector
    mp_det = MediaPipeDetector(min_detection_confidence=0.5)
    cnn_det = None
else:
    cnn_det = load_cnn_detector(opt)
    mp_det = None

print("Loading Classifier...")
classifier = load_classifier(opt)
print("Initialization Complete.")

Loading Detector...


AttributeError: 'Namespace' object has no attribute 'pretrain_path'